# FEM Plate-with-Hole GNN Surrogate — Kaggle GPU setup

Sibling notebook to `colab_setup.ipynb`, adapted for Kaggle instead of Colab.
Much simpler than the AirfRANS project's Kaggle notebook: that one had to
stream-extract a ~15GB external dataset one case at a time to fit Kaggle's
disk quota. This dataset (200 Abaqus cases, ~207MB raw + ~78MB cached) is
small enough to already be committed to the repo -- clone and go, no
download/streaming step needed.

No Drive-equivalent live mount here: Kaggle persists across sessions via
**Datasets** (read-only, attached to a session) and a notebook's own
**Output** (via "Save Version"), not a synced folder -- checkpoints below go
to `/kaggle/working/`, which becomes that notebook's Output when you save a
version.

Before running: Settings (right sidebar) > Accelerator > GPU, and
Internet > On (needed for git clone / pip install).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the repo

Pushed to `https://github.com/Revanthkr1/fem-plate-gnn` (**private**) --
`data/raw/*.json` (200 cases) and `data/norm_stats.npz` are committed, so they
arrive with the clone; only `data/cache/` (gitignored, rebuilt in section 2)
is missing after this.

Because the repo is private, plain `git clone` would prompt for credentials
and fail non-interactively. The cell below reads a GitHub personal access
token from Kaggle's **Secrets** add-on instead of hardcoding one -- add a
token there first: notebook menu > Add-ons > Secrets, named `GITHUB_TOKEN`
(generate one, classic, `repo` scope, at
https://github.com/settings/tokens). Never paste a real token directly into
this notebook -- it gets committed to git.

(Alternative: make the repo public later once you're ready to share it, and
plain `git clone` works with no token at all.)

In [ ]:
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{token}@github.com/Revanthkr1/fem-plate-gnn.git"

!git clone $REPO_URL repo
%cd repo

## 2. Install dependencies + rebuild the cache

Kaggle ships torch with CUDA already installed -- don't reinstall it.
`torch_geometric` installs as pure Python (same `torch_geometric.utils.scatter`-only
usage as AirfRANS's model.py, ported unchanged). No `airfrans` package needed
-- this project doesn't depend on it.

`data/cache/` (preprocessed graph tensors) is gitignored and cheap to rebuild
-- these are small meshes (~5-9k nodes depending on hole count, not
AirfRANS's ~180k), so this is seconds. Safe to re-run: `preprocess_case`
skips any case already cached. 400 cases as of phase 11 (200 single-hole +
200 with variable hole count 0-3).

In [ ]:
!pip install -q torch_geometric lightning pyvista pyyaml

In [ ]:
import glob
import os

from src.preprocess import preprocess_split

RAW_DIR = "data/raw"
CACHE_DIR = "/kaggle/working/cache"

n_cases = len(glob.glob(os.path.join(RAW_DIR, "case_*.json")))
case_ids = list(range(n_cases))
preprocess_split(RAW_DIR, case_ids, CACHE_DIR)
print(f"cached {len(glob.glob(os.path.join(CACHE_DIR, 'case_*.pt')))}/{n_cases} cases")

## 3. Smoke test: one case on the real GPU (optional)

Same forward/backward timing check as the Colab notebook's section 4 --
confirms the model + a real cached graph actually move to the GPU and run
before committing to a full training loop. Skip if you're just resuming a
training run.

In [ ]:
import time
import numpy as np
import torch

from src.dataset import CachedPyGPlateHoleDataset
from src.model import MeshGraphNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

stats = dict(np.load("data/norm_stats.npz"))
ds = CachedPyGPlateHoleDataset(CACHE_DIR, [0], stats=stats)
data = ds[0].to(device)

model = MeshGraphNet(node_in_dim=3, edge_in_dim=2, out_dim=3).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

t0 = time.time()
pred = model(data.x, data.edge_index, data.edge_attr)
loss = torch.nn.functional.mse_loss(pred, data.y)
loss.backward()
opt.step()
print(f"nodes={data.x.shape[0]}, edges={data.edge_index.shape[1]}, "
      f"forward+backward+step={time.time()-t0:.2f}s, loss={loss.item():.4f}")

## 4. Train

**Phase 11: the actual geometric-generalization test.** Cases 200-399 vary
hole *count* (0-3), not just hole radius/position within one template. Every
case with exactly 3 holes (51 cases) is held out of training *entirely* --
`src/splits.py::hole_count_splits()` -- so evaluating on them afterward tests
whether the model generalizes to a hole count it has genuinely never seen,
not just a new radius/position combination. A normal random 20-case
in-distribution holdout is drawn from the remaining (0/1/2-hole) cases, same
role as the phase 8-10b validation split. `data/norm_stats.npz` was
recomputed over the training-distribution cases only (excludes the held-out
count=3 bucket, so normalization can't leak any information about it).

Architecture is the phase-10b-validated one (`node_in_dim=3`, `[x, y, load]`
only -- no hand-fed hole geometry; `n_message_passing=8`) which beat every
previous result including the hand-engineered-feature version (3.9%
peak-stress error, 4.9mm location error on its own held-out split -- see
`PROJECT_FLOW.md` phase 10b). This run doesn't change the architecture,
just the training data and split.

Checkpoints go in their own subdirectory (`/kaggle/working/phase11/`) --
learned the hard way in phase 9/10 that reusing a directory (even under a
different final filename) lets `train.py`'s auto-resume silently find and
load an unrelated run's periodic checkpoint.

Click **Save Version** (periodically, or once done) to persist checkpoints
past this session. Once this finishes, bring the checkpoint back for
evaluation on *both* splits -- the normal held-out set and, more importantly,
the held-out count=3 cases -- since that comparison is the actual point of
this phase.

In [ ]:
import os
import yaml

from src.splits import hole_count_splits
from src.train import main as train_main

config = yaml.safe_load(open("configs/base.yaml"))

# Held-out count=3 cases never enter training at all -- that's the actual
# generalization test. n_val below matches len(splits["val"]) exactly, so
# train.py's existing case_ids[-n_val:] slicing lands on the right cases.
splits = hole_count_splits("data/raw", held_out_count=3, n_val=20, seed=0)
case_ids = splits["train"] + splits["val"]
print(f"train={len(splits['train'])}, val={len(splits['val'])}, "
      f"held-out count=3 (not used here)={len(splits['test_ood'])}")

# Own subdirectory, not just a different filename -- see section 4 markdown.
RUN_DIR = "/kaggle/working/phase11"
os.makedirs(RUN_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(RUN_DIR, "meshgraphnet_phase11.ckpt")

train_main(
    cache_dir=CACHE_DIR,
    stats_path="data/norm_stats.npz",
    checkpoint_path=CHECKPOINT_PATH,
    case_ids=case_ids,
    model_kwargs=config["model"],
    max_epochs=config["training"]["max_epochs"],
    batch_size=config["training"]["batch_size"],
    accumulate_grad_batches=config["training"]["accumulate_grad_batches"],
    n_val=len(splits["val"]),
    lr=config["training"]["lr"],
    checkpoint_every_n_epochs=config["training"]["checkpoint_every_n_epochs"],
    num_workers=config["training"]["num_workers"],
    precision="16-mixed",  # GPU-specific override -- base.yaml's 32-true is for local CPU runs
)